# Reproduce the JPMedAI revision (imports `analysis.*`)

First cell: seed 42, repository paths, `uv` kernel.

- Kernel name: **`sahel-malaria`** (the project `.venv`).
- Create it once: `uv sync` then `uv run python -m ipykernel install --user --name sahel-malaria --display-name "Python (sahel-malaria)"`.
- This notebook **imports** the `analysis` package. It does **not** overwrite `outputs/tables/metrics_loco_primary.csv` or regenerate Figures 1–5.
- Self-contained inlined source is in `full_pipeline.ipynb`.

In [ ]:
from pathlib import Path
import os
import sys

SEED = 42
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "analysis").is_dir() else HERE.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT", ROOT)
print("SEED", SEED)
print("kernel note: select sahel-malaria (uv .venv), not a system Python")
print("python", sys.executable)

In [ ]:
from analysis.config import DATA_DIR, FIGURES, LGBM_GRID, PRIMARY_FEATURES, SEED as CFG_SEED, TABLES, ensure_dirs
from analysis.figures_tables import run_figures, table1_expanded, table_s3_subgroups, figure_s1_workflow
from analysis.models_loco import run_loco
from analysis.sensitivity import (
    antimalarial_exclusion,
    equal_country_weight_loco,
    recalibration_sensitivity,
)

import pandas as pd

assert CFG_SEED == 42
ensure_dirs()
print("PRIMARY_FEATURES", PRIMARY_FEATURES)
print("SHAP / sensitivity LightGBM grid point", LGBM_GRID[1])

## Locked primary Table 2

Load saved leave-one-country-out metrics. `run_loco(..., tag="primary")` refuses to overwrite the CSV.

In [ ]:
primary = pd.read_csv(TABLES / "metrics_loco_primary.csv")
print(primary[primary.model == "lightgbm"][["country", "n", "rdt_pos", "auc", "oe"]].to_string(index=False))
loaded = run_loco(tag="primary", overwrite=False)
assert (TABLES / "metrics_loco_primary.csv").exists()
print("rows in loaded primary metrics", len(loaded["metrics"]))

## New revision analyses

Equal-country-weight LOCO (training folds only), intercept/Platt recalibration on a 50/50 split inside each held-out country, antimalarial exclusion counts, expanded Table 1, Table S3, Figure S1.

In [ ]:
analytic = pd.read_parquet(DATA_DIR / "analytic.parquet") if (DATA_DIR / "analytic.parquet").exists() else None
excl = antimalarial_exclusion(analytic)
display(excl.tail(1))
eq = equal_country_weight_loco(analytic)
display(eq[eq.model == "lightgbm"][["country", "auc", "oe"]])
recal = recalibration_sensitivity()
display(recal)

In [ ]:
if analytic is not None:
    tab1 = table1_expanded(analytic)
    display(tab1[["country", "n", "rdt_pos", "weighted_rdt_prev", "age_months", "female", "urban", "net_used", "age_months_miss_pct"]])
s3 = table_s3_subgroups()
print("Table S3 rows", len(s3))
figure_s1_workflow()
run_figures(regenerate_primary=False)
print("Figure S1", FIGURES / "figure_s1_workflow.png")